# SEACOFS Eddy Dataset Pipeline

This notebook is a concise control panel for the modular SEACOFS eddy dataset pipeline. The scientific functions live in `src/seacofs_eddy_dataset/`; this notebook only loads config, runs stages, and checks outputs.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT

## Load Config

Copy `config/example.yaml` to `config/local.yaml` and edit paths before running the pipeline.

In [ ]:
from seacofs_eddy_dataset.config import load_config
from seacofs_eddy_dataset.pipeline import STAGES, run_stage, run_all

CONFIG_PATH = PROJECT_ROOT / "config" / "local.yaml"
config = load_config(CONFIG_PATH)

config.output_root

In [ ]:
[(stage.name, stage.description) for stage in STAGES]

## Run Individual Stages

Run any single stage while developing or resuming part of the workflow.

In [ ]:
run_stage("detect_nencioli", config)

## Run The Workflow

Edit this list if you want to resume from a later stage or stop before analysis.

In [ ]:
STAGES_TO_RUN = [
    "detect_nencioli",
    "fit_doppio_surface",
    "track_eddies",
    "process_tracked_dataset",
    "compute_vertical_profiles",
    "qc_vertical_profiles",
    "compute_tilt",
    "analyse_tilt",
]

for stage_name in STAGES_TO_RUN:
    run_stage(stage_name, config)

## Inspect Outputs

In [ ]:
import pandas as pd

output_dirs = {
    "detections": config.output_root / "detections",
    "surface_eddies": config.output_root / "surface_eddies",
    "tracked": config.output_root / "tracked",
    "processed": config.output_root / "processed",
    "vertical_profiles": config.output_root / "vertical_profiles",
    "vertical_profiles_confirmed": config.output_root / "vertical_profiles_confirmed",
    "tilt": config.output_root / "tilt",
    "analysis": config.output_root / "analysis",
}

pd.DataFrame(
    [
        {"stage": name, "path": str(path), "parquet_files": len(list(path.glob("*.parquet"))) if path.exists() else 0}
        for name, path in output_dirs.items()
    ]
)

In [ ]:
detection_files = sorted((config.output_root / "detections").glob("fnumber=*.parquet"))

if detection_files:
    df_detect = pd.concat((pd.read_parquet(path) for path in detection_files), ignore_index=True)
    display(df_detect.head())
    display(df_detect.groupby("Cyc").size().rename("count"))
    display(df_detect.groupby("fnumber").size().rename("detections").describe())
else:
    print(f"No detection files found in {config.output_root / 'detections'}")

## Run Everything

Equivalent shortcut for the full sequence.

In [ ]:
# run_all(config)